In [0]:
# Import configuration and helper functions
%run config.py
%run resources/data_setup_functions

import sys
import time
import logging
import os
from pathlib import Path

from databricks.sdk import WorkspaceClient
from databricks.sdk.runtime import dbutils
# from databricks.vector_search.client import VectorSearchClient

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

# Step 1: Create catalog & schema
create_catalog_and_schema(spark, catalog, schema)

# Step 2: Create volume for raw documents
create_volume(spark, catalog, schema, volume_name)

# Step 3: Create table for document metadata
create_table(spark, catalog, schema, table)

# Step 4: Upload first file from data/10k folder
data_folder = "/data"
local_file_path = os.getcwd()+data_folder
local_root = local_file_path  # path to "data/"

upload_file_to_volume(spark, local_file_path, catalog, schema, volume_name, table)

# Step 5: Create tables for volume subfolders (inline implementation)
logger.info("Creating tables for volume subfolders...")

volume_path = f"/Volumes/{catalog}/{schema}/{volume_name}"
subfolders = dbutils.fs.ls(volume_path)

create_tables_from_volume_subfolders(spark, catalog, schema, volume_name, logger)

'''
for item in subfolders:
    if item.isDir():
        subfolder_name = item.name.rstrip('/')
        table_name = f"{subfolder_name}_data"
        subfolder_path = item.path
        
        logger.info(f"Processing subfolder: {subfolder_name}")
        
        # Check if subfolder contains CSV files
        files = dbutils.fs.ls(subfolder_path)
        csv_files = [f for f in files if f.name.endswith('.csv')]
        
        if csv_files:
            # Load CSV files from volume into table
            logger.info(f"  Found {len(csv_files)} CSV file(s), creating table {table_name}")
            csv_path = f"{subfolder_path}*.csv"
            df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(csv_path)
            full_table_name = f"{catalog}.{schema}.{table_name}"
            df.write.mode("overwrite").saveAsTable(full_table_name)
            logger.info(f"  ✓ Created table {full_table_name} with {df.count()} rows")
        else:
            # For PDF/document folders, create a table with file metadata using read_files
            logger.info(f"  Found document files, creating metadata table {table_name}")
            doc_path = f"{subfolder_path}*"
            
            # Create table with file metadata
            df = spark.sql(f"""
                SELECT 
                    _metadata.file_path as file_path,
                    _metadata.file_name as file_name,
                    _metadata.file_size as file_size,
                    _metadata.file_modification_time as file_modification_time,
                    '{subfolder_name}' as document_category
                FROM read_files('{doc_path}')
            """)
            
            full_table_name = f"{catalog}.{schema}.{table_name}"
            df.write.mode("overwrite").saveAsTable(full_table_name)
            logger.info(f"  ✓ Created metadata table {full_table_name} with {df.count()} files")

logger.info("\n✅ All subfolder tables created successfully!")
logger.info("All setup steps completed successfully.")
'''